# Drone Rescue v0 - Static Benchmark

This run evaluates A* and Dijkstra on seeded, unseen static maps.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from drone_rescue.evaluation.benchmark import generate_map_suite, run_static_benchmark
from drone_rescue.planners.astar import astar
from drone_rescue.planners.world_model import GridWorldModel
from drone_rescue.visualization.replay import render_grid, render_path


In [2]:
EPISODES = 100
SIZE = 10
OBSTACLE_PROBABILITY = 0.2
SEED = 42

maps = generate_map_suite(EPISODES, SIZE, OBSTACLE_PROBABILITY, SEED)
results = run_static_benchmark(maps)
results_frame = pd.DataFrame([result.__dict__ for result in results])
display(results_frame)

,planner,success_rate,average_path_length,optimal_path_length,average_cost,planning_latency_ms,path_efficiency,episodes
0,astar,1.0,7.73,7.73,7.73,0.178473,1.0,100
1,dijkstra,1.0,7.73,7.73,7.73,0.362651,1.0,100


In [3]:
output_dir = Path("results/static_v0")
output_dir.mkdir(parents=True, exist_ok=True)
results_frame.to_json(output_dir / "metrics.json", orient="records", indent=2)
results_frame.to_csv(output_dir / "metrics.csv", index=False)
print(f"Saved results to {output_dir.resolve()}")

Saved results to C:\Users\ujwal\OneDrive\Documents\Github\Reinforcement-learning\RL_LLM\0_what_is_RL\02_1_DQN_DRONE\experiments\results\static_v0


In [4]:
sample = maps[0]
model = GridWorldModel(sample.obstacles)
path, cost, _ = astar(model, sample.start, sample.rescue)

print("Original grid")
print(render_grid(sample.obstacles, sample.start, sample.rescue))
print("\nA* path")
print(render_path(sample.obstacles, path, sample.start, sample.rescue))
print(f"\nPath length: {len(path) - 1}, cost: {cost}")

Original grid
+---------------------+
| . . . . X . . . X . |
| . . . . . . . X . . |
| . . . . . X . X X . |
| . . . . R X X . . . |
| . . . . . . . . . X |
| X X . . . . . . X X |
| . . . . . . . . X . |
| . . . . X . . . . . |
| . . . X X X . . X S |
| X . . . . . . X X . |
+---------------------+

A* path
+---------------------+
| . . . . X . . . X . |
| . . . . . . . X . . |
| . . . . . X . X X . |
| . . . . R X X . . . |
| . . . . o o o o . X |
| X X . . . . . o X X |
| . . . . . . . o X . |
| . . . . X . . o o o |
| . . . X X X . . X S |
| X . . . . . . X X . |
+---------------------+

Path length: 10, cost: 10.0
